# Opening range short trades: RSI at breakout

This tests RSI as a filter on the **saved 30-minute opening-range OR-line limit** trades with a **1% stop and 2% profit target**. The previous grid already applied: QQQ opening gap within ±1%, ticker gap down more than 1%, entry before noon with QQQ below its 9:30 open immediately before entry, one pending order or open trade per ticker, 15-minute order life, 3:55 ET exit, 5 bps modeled slippage each side, and November 4–5 exclusion.

RSI(14) is calculated from completed regular-session 5-minute ticker candles, carrying Wilder-smoothed RSI across available trading days. The value and one-bar change are measured at the **breakout signal**, before any later OR-line limit fill. Missing RSI remains visible. No RSI threshold affects the original trade replay. Winners reach the full 2% target; stop hits or time exits at or below −0.5% are losses; other time exits are neutral. Compare every band with the unfiltered baseline and repeat excluding November 20 and October 12, 14, 16. This is exploratory and uses modeled OHLCV fills.

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd, numpy as np, re
drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/backtesting'),Path('/content/drive/MyDrive/Backtesting')] if p.is_dir()),None)
if ROOT is None: raise FileNotFoundError('Set ROOT to your Backtesting folder')
OUT=Path('/content/drive/MyDrive/Opening_Range_RSI_Filter');OUT.mkdir(exist_ok=True)
SLIPPAGE_BPS=5;WAIT_MIN=15
EXCLUDED={'2026-11-04','2026-11-05'}
STANDOUT={'2026-10-12','2026-10-14','2026-10-16'}
files={}
for p in ROOT.rglob('*_1m_*.csv'):
    m=re.match(r'(.+)_1m_\d{4}-\d{2}-\d{2}',p.stem)
    if m: files.setdefault(m.group(1).upper(),[]).append(p)
if 'QQQ' not in files:raise FileNotFoundError('QQQ 1-minute bars missing')
print('Ticker symbols with 1m files:',len(files),'| CSV files:',sum(map(len,files.values())))

def load_sessions(paths):
    groups={}
    for p in paths:
        try:
            a=pd.read_csv(p)
            col=next((c for c in a if c.lower() in ('datetime','date','timestamp')),a.columns[0])
            a.index=pd.to_datetime(a[col],utc=True,errors='coerce').dt.tz_convert('America/New_York')
            a=a[['Open','High','Low','Close','Volume']].apply(pd.to_numeric,errors='coerce').dropna(subset=['Open','High','Low','Close'])
            a=a.between_time('09:30','15:59')
            for date,g in a.groupby(a.index.date): groups.setdefault(str(date),[]).append(g)
        except Exception as e: print('Skipped',p.name,str(e)[:100])
    out={}
    for date,parts in groups.items():
        a=pd.concat(parts).sort_index();out[date]=a[~a.index.duplicated(keep='last')]
    return out

qqq=load_sessions(files['QQQ'])
market_dates=sorted(qqq)
prior_market={market_dates[j]:market_dates[j-1] for j in range(1,len(market_dates))}
def open_of(d,date):
    t=pd.Timestamp(date+' 09:30',tz='America/New_York')
    return float(d.loc[t,'Open']) if d is not None and t in d.index else np.nan
def close_of(d,date):
    if d is None or date is None:return np.nan
    t=pd.Timestamp(date+' 15:59',tz='America/New_York')
    return float(d.loc[t,'Close']) if d is not None and t in d.index else np.nan
def gap(op,prev):return 100*(op/prev-1) if np.isfinite(op) and np.isfinite(prev) and prev>0 else np.nan
qqq_gap={day:gap(open_of(qqq[day],day),close_of(qqq.get(prior_market.get(day)),prior_market.get(day))) for day in market_dates}
print('QQQ market dates:',len(market_dates))
SESSION_GAPS=Path('/content/drive/MyDrive/Opening_Range_All_Tickers_QQQ_At_Entry/all_ticker_session_gaps.csv')
if not SESSION_GAPS.exists():raise FileNotFoundError(SESSION_GAPS)
sessions=pd.read_csv(SESSION_GAPS,dtype={'ticker':str,'date':str})
chosen=sessions[sessions.qqq_gap_pct.abs().le(1)&sessions.ticker_gap_pct.lt(-1)&
                ~sessions.date.isin(EXCLUDED)].copy()
needed=chosen.groupby('ticker').date.apply(set).to_dict()
print('Qualifying ticker-days:',len(chosen),'| tickers:',len(needed))
if not needed:raise RuntimeError('No ticker-days pass saved opening gap filters')

In [ ]:
def breakout_events(d,date,minutes):
    start=pd.Timestamp(date+' 09:30',tz='America/New_York')
    opening=d.loc[(d.index>=start)&(d.index<start+pd.Timedelta(minutes=30))]
    hi=float(opening.High.max());lo=float(opening.Low.min())
    if minutes==1:bars=d
    else:
        bars=d.resample('2min',origin=start,label='left',closed='left').agg({'Open':'first','High':'max','Low':'min','Close':'last'})
        count=d.Close.resample('2min',origin=start,label='left',closed='left').count()
        bars=bars[count.eq(2)].dropna(subset=['Open','Close'])
    out=[];previous={'long':False,'short':False}
    for t,c in bars.iterrows():
        if t<start+pd.Timedelta(minutes=30):continue
        signal=t+pd.Timedelta(minutes=minutes)
        if signal.strftime('%H:%M')>'15:00':break
        up=bool(c.Close>hi);down=bool(c.Close<lo)
        if up and not previous['long']:out.append(dict(signal=signal,side='long',boundary=hi,or_high=hi,or_low=lo))
        if down and not previous['short']:out.append(dict(signal=signal,side='short',boundary=lo,or_high=hi,or_low=lo))
        previous={'long':up,'short':down}
    return out

def next_bar(d,signal):
    i=d.index.searchsorted(signal)
    return d.index[i] if i<len(d) and d.index[i]<=signal+pd.Timedelta(minutes=1) else None

def limit_entry(d,e):
    t=next_bar(d,e['signal'])
    if t is None:return None,'missing_next_open',e['signal']
    sign=1 if e['side']=='long' else -1
    price=e['boundary'];end=e['signal']+pd.Timedelta(minutes=WAIT_MIN)
    for q,c in d.loc[(d.index>=t)&(d.index<end)].iterrows():
        if sign*(float(c.Open)-price)<=0:
            return (q,float(c.Open),'open'),'filled',q
        if c.Low<=price if sign==1 else c.High>=price:
            return (q,price,'wick'),'filled',q
    return None,'unfilled_15m',end

def exit_trade(d,date,e,fill,target_pct,stop_pct):
    t,raw,fill_type=fill;sign=1 if e['side']=='long' else -1
    entry=raw*(1+sign*SLIPPAGE_BPS/10000)
    stop=entry*(1-sign*stop_pct);target=entry*(1+sign*target_pct)
    end=pd.Timestamp(date+' 15:55',tz='America/New_York')
    path=d.loc[(d.index>=t)&(d.index<=end)]
    if path.empty:return None
    exit_time=path.index[-1];exit_raw=float(path.iloc[-1].Close);outcome='time'
    for q,c in path.iterrows():
        bad=c.Low<=stop if sign==1 else c.High>=stop
        good=(c.High>=target if sign==1 else c.Low<=target) and not (q==t and fill_type=='wick')
        if bad or good:
            exit_time=q;exit_raw=stop if bad else target;outcome='stop' if bad else 'target';break
    exit_price=exit_raw*(1-sign*SLIPPAGE_BPS/10000)
    R=sign*(exit_price-entry)/entry/stop_pct
    result='win' if outcome=='target' else 'loss' if outcome=='stop' or R*stop_pct<=-.005 else 'neutral'
    return dict(entry_time=t,entry=entry,exit_time=exit_time,exit=exit_price,R=R,outcome=outcome,result=result,
                raw_fill_price=raw,fill_type=fill_type)

def price_group(price):
    if price<50:return 'Under $50'
    if price<100:return '$50–<$100'
    if price<200:return '$100–<$200'
    return '$200+'

In [ ]:
TRADES=Path('/content/drive/MyDrive/Opening_Range_Stop_Target_Grid/short_grid_with_qqq_entry.csv')
if not TRADES.exists():raise FileNotFoundError(TRADES)
all_short=pd.read_csv(TRADES,dtype={'ticker':str,'date':str})
base=all_short[all_short.entry_method.eq('OR_line_limit') &
               np.isclose(all_short.stop_pct,.01) & np.isclose(all_short.target_pct,.02) &
               all_short.before_noon.eq(True) & all_short.qqq_falling.eq(True)].copy()
if base.empty:raise RuntimeError('No baseline trades; check the saved stop/target grid output')
print('Unfiltered baseline:',len(base),'W/L/N:',
      *[int(base.result.eq(v).sum()) for v in ('win','loss','neutral')],
      '| net R:',round(base.R.sum(),2),'| tickers:',base.ticker.nunique(),'| dates:',base.date.nunique())
print('Expected previous snapshot: 610 trades, 303/246/61, +369R; changes can reflect refreshed source data.')
base['signal_et']=pd.to_datetime(base.signal_time,utc=True).dt.tz_convert('America/New_York')

def completed_five_minute_bars(stock):
    parts=[]
    for day,d in sorted(stock.items()):
        start=pd.Timestamp(day+' 09:30',tz='America/New_York')
        x=d.loc[(d.index>=start)&(d.index<start+pd.Timedelta(hours=6,minutes=30))]
        if x.empty:continue
        price=x.Close.resample('5min',origin=start,label='right',closed='left').last()
        count=x.Close.resample('5min',origin=start,label='right',closed='left').count()
        parts.append(price[count.eq(5)])
    return pd.concat(parts).sort_index() if parts else pd.Series(dtype=float)

def wilder_rsi(closes,n=14):
    delta=closes.diff()
    up=delta.clip(lower=0).ewm(alpha=1/n,adjust=False,min_periods=n).mean()
    down=(-delta.clip(upper=0)).ewm(alpha=1/n,adjust=False,min_periods=n).mean()
    rsi=100-100/(1+up/down.replace(0,np.nan))
    rsi=rsi.mask(down.eq(0)&up.gt(0),100).mask(down.eq(0)&up.eq(0),50)
    return rsi

base['rsi_5m_14']=np.nan;base['rsi_change_5m']=np.nan
for n,(ticker,positions) in enumerate(base.groupby('ticker').groups.items(),1):
    if ticker not in files:continue
    closes=completed_five_minute_bars(load_sessions(files[ticker]))
    if len(closes)<15:continue
    rsi=wilder_rsi(closes)
    for row_idx in positions:
        signal=base.at[row_idx,'signal_et']
        k=rsi.index.searchsorted(signal,side='right')-1
        if k<1:continue
        if pd.notna(rsi.iloc[k]):base.at[row_idx,'rsi_5m_14']=float(rsi.iloc[k])
        if pd.notna(rsi.iloc[k]) and pd.notna(rsi.iloc[k-1]):
            base.at[row_idx,'rsi_change_5m']=float(rsi.iloc[k]-rsi.iloc[k-1])
    if n%100==0:print('Processed tickers:',n,'/',base.ticker.nunique())
base['rsi_band']=pd.cut(base.rsi_5m_14,[-np.inf,30,40,50,60,70,np.inf],
    labels=['<30','30–<40','40–<50','50–<60','60–<70','70+'],right=False)
base['rsi_direction']=np.select([base.rsi_change_5m.lt(0),base.rsi_change_5m.gt(0),
    base.rsi_change_5m.eq(0)],['falling','rising','flat'],default='missing')
base.to_csv(OUT/'baseline_trades_with_signal_rsi.csv',index=False)
print('Trades with signal-time RSI:',base.rsi_5m_14.notna().sum(),'/',len(base))
print('Missing RSI by date:')
display(base.assign(missing=base.rsi_5m_14.isna()).groupby('date').missing.agg(['sum','count']))

In [ ]:
def summarize(df,group):
    out=df.assign(win=df.result.eq('win').astype(int),loss=df.result.eq('loss').astype(int),
                  neutral=df.result.eq('neutral').astype(int)).groupby(group,observed=True,dropna=False).agg(
        trades=('R','size'),wins_2pct=('win','sum'),losses=('loss','sum'),neutrals=('neutral','sum'),
        net_R=('R','sum'),tickers=('ticker','nunique'),dates=('date','nunique')).reset_index()
    out['win_rate_pct']=(100*out.wins_2pct/out.trades).round(1)
    out['mean_R']=(out.net_R/out.trades).round(3)
    assert (out.wins_2pct+out.losses+out.neutrals==out.trades).all()
    return out.round({'net_R':2})
removed={'2026-11-20','2026-10-12','2026-10-14','2026-10-16'}
for label,data in [('all_dates',base),('without_four_outlier_dates',base[~base.date.isin(removed)])]:
    for suffix,group in [('bands',['rsi_band']),('band_direction',['rsi_band','rsi_direction']),
                         ('direction',['rsi_direction'])]:
        result=summarize(data,group)
        result.to_csv(OUT/(label+'_'+suffix+'.csv'),index=False)
        print('\n'+label+' | '+suffix);display(result)
    print(label+' | baseline W/L/N and net R:')
    print(len(data),*[int(data.result.eq(x).sum()) for x in ['win','loss','neutral']],round(data.R.sum(),2))
    print(label+' | RSI available W/L/N and net R:')
    available=data[data.rsi_5m_14.notna()]
    print(len(available),*[int(available.result.eq(x).sum()) for x in ['win','loss','neutral']],round(available.R.sum(),2))
print('Saved outputs in',OUT)